---
title: "Chương 6: Lỗi và hạn chế"
short_title: "6. Lỗi và hạn chế"
---

Chương này tổng hợp kết quả kiểm tra tự động của `validate.py` và các trường hợp crawl chưa trọn vẹn. Ý nghĩa các mức kiểm tra:

| Mức | Ý nghĩa |
| :--- | :--- |
| `PASS` | Kiểm tra đó đạt |
| `WARN` | Cần giải thích hoặc xem lại thủ công, ví dụ ngày nộp arXiv của reference còn thiếu, phiên bản 404 đã được kiểm tra tay |
| `PENDING` | Việc chưa xong, ví dụ phiên bản `failed` chờ chạy lại, bài chưa có file references do Semantic Scholar trả 429 |
| `FAIL` | Lỗi cấu trúc hoặc nội dung: thiếu file, file rỗng, dữ liệu reference không nhất quán |

In [ ]:
import pandas as pd
import plotly.express as px
from report_utils import (COLORS, FINAL_SOURCE, REF_OK, group, human_bytes, human_seconds,
                          load, members, meta, metric, mib, pct, provenance, show, table)
provenance()

## 1. Kết quả kiểm tra tự động

In [ ]:
vc = load("validation_counts")
pivot = vc.pivot_table(index="check_name", columns="status", values="count", aggfunc="sum", fill_value=0)
pivot = pivot.reindex(columns=[c for c in ["PASS", "WARN", "PENDING", "FAIL"] if c in pivot.columns])
pivot.columns.name = None
table(pivot.reset_index().rename(columns={"check_name": "Kiểm tra"}), "Bảng 6.1. Số kết quả theo kiểm tra và mức")
bad = vc[vc["status"] != "PASS"].groupby(["check_name", "status"])["count"].sum().reset_index()
if len(bad):
    fig = px.bar(bad, x="count", y="check_name", color="status", orientation="h",
                 color_discrete_map={"WARN": "#DDAA33", "PENDING": "#4477AA", "FAIL": "#BB5566"},
                 title="Các kiểm tra chưa đạt (không tính PASS)",
                 labels={"count": "Số kết quả", "check_name": "Kiểm tra", "status": "Mức"})
    show(fig)
else:
    print("Mọi kiểm tra đều PASS.")

## 2. Các trường hợp crawl chưa trọn vẹn

In [ ]:
papers, versions, m = load("papers"), load("versions"), members()
found = papers[papers["discovery_status"] == "success"]
cases = [
    ("Discovery", "ID không tồn tại trên arXiv (not_found)", int(m["ids_not_found"].sum())),
    ("Discovery", "ID còn lỗi API (api_error)", int(m["ids_api_error"].sum())),
    ("Mã nguồn", "Phiên bản arXiv trả 404 (not_found)", int((versions["status"] == "not_found").sum())),
    ("Mã nguồn", "Phiên bản chỉ có PDF (pdf_only)", int((versions["status"] == "pdf_only").sum())),
    ("Mã nguồn", "Phiên bản DOCX/PostScript (no_tex)", int((versions["status"] == "no_tex").sum())),
    ("Mã nguồn", "Phiên bản còn lỗi tạm thời (failed)", int((versions["status"] == "failed").sum())),
    ("Mã nguồn", "Phiên bản chưa có trạng thái (missing)", int((versions["status"] == "missing").sum())),
    ("References", "Bài chưa được Semantic Scholar lập chỉ mục (not_found)", int((found["ref_status"] == "not_found").sum())),
    ("References", "Bài có reference nhưng không chia sẻ (unavailable)", int((found["ref_status"] == "unavailable").sum())),
    ("References", "Bài chưa có file references (lỗi API ở lần chạy cuối)", int((found["ref_status"] == "missing").sum())),
    ("References", "Bài chưa đủ ngày nộp arXiv của reference",
     int(found["arxiv_dates_complete"].astype(str).eq("False").sum())),
]
table(pd.DataFrame(cases, columns=["Bước", "Trường hợp", "Số lượng"]), "Bảng 6.2. Các trường hợp chưa trọn vẹn, cả nhóm")

## 3. Ví dụ các kết quả chưa đạt

Tối đa 30 dòng, ưu tiên `FAIL` rồi `PENDING`, `WARN` (danh sách đầy đủ nằm trong `validation_report.csv` của từng thành viên).

In [ ]:
issues = load("validation_issues")
order = {"FAIL": 0, "PENDING": 1, "WARN": 2}
if len(issues):
    issues = issues.assign(o=issues["status"].map(order)).sort_values(["o", "member", "arxiv_id"]).drop(columns="o")
    table(issues.head(30).fillna(""), "Bảng 6.3. Ví dụ kết quả kiểm tra chưa đạt")
else:
    print("Không có kết quả nào khác PASS.")
w = meta()["warnings"]
if w:
    print("Cảnh báo khi xuất dữ liệu báo cáo:")
    for line in w:
        print(" -", line)

## 4. Nguyên nhân và cách xử lý

| Vấn đề | Nguyên nhân | Cách nhóm xử lý |
| :--- | :--- | :--- |
| arXiv trả HTTP 429/503 khi tải nhiều | Giới hạn tải của arXiv, nhiều thành viên crawl cùng lúc | Tạm dừng mọi request tới arXiv với thời gian chờ tăng dần; lượt retry sau cho các phiên bản `failed` |
| Phiên bản trả 404 | Phiên bản bị rút hoặc arXiv không lưu mã nguồn | Ghi trạng thái `not_found`, không retry; kiểm tra tay một số trường hợp trên trang `arxiv.org/abs/<id><version>` |
| Phiên bản chỉ có PDF, DOCX | Tác giả không nộp mã nguồn TeX | Giữ thư mục rỗng và trạng thái `pdf_only`/`no_tex`, không bỏ bài |
| Semantic Scholar trả 429 | Giới hạn request của API | API key riêng, gom nhiều bài trong một lời gọi `/paper/batch`, retry có chờ; bài lỗi không bị ghi thành "không có reference" |
| Bài mới chưa có trên Semantic Scholar | Semantic Scholar lập chỉ mục chậm vài ngày sau khi bài lên arXiv | Ghi `not_found`; có thể chạy lại `references.py --retry not_found` sau |
| Thiếu ngày nộp arXiv của reference | Cần thêm một lời gọi arXiv API cho mỗi 100 ID được trích dẫn | Lấy ngày từ bản metadata arXiv trên Kaggle (`ref_dates_kaggle.py`) để không tốn request arXiv |
| Colab ngắt giữa chừng | Giới hạn thời gian của runtime | Checkpoint theo từng phiên bản, chạy lại cùng lệnh để làm tiếp |

## 5. Hạn chế

- **Thời gian chạy:** do Colab bị ngắt, thời gian của một thành viên là tổng của nhiều lần chạy, có thêm chi phí khởi động lại, không phải một lần chạy liền mạch từ thư mục trống.
- **Đo RAM và đĩa bằng lấy mẫu:** đỉnh RAM ngắn hơn 2 giây hoặc đỉnh dung lượng giữa hai lần quét 300 giây có thể bị bỏ sót. Thư mục dữ liệu nằm trên Google Drive nên số đo dung lượng còn chịu độ trễ đồng bộ của Drive.
- **References phụ thuộc Semantic Scholar:** độ phủ và chất lượng metadata của reference là của Semantic Scholar; nhóm không tự điền thông tin không kiểm chứng được.
- **Danh sách ID:** số ID được giao được đọc từ manifest của từng thành viên; muốn đối chiếu chặt với danh sách chính thức thì chạy `validate.py --assigned-ids` với file ID của giảng viên.